In [0]:
%pip install fitparse

In [0]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from utils.triathlon_utils import (
    parse_fit, 
    clean_fit_data, 
    calculate_elapsed_time, 
    apply_triathlon_style,
    format_time_axis,
    export_for_instagram,
    KLEUREN_PALETTE
)

from utils.config import DEFAULT_ATHLETE, DEFAULT_PATHS

In [0]:
def analyze_slope_vs_cadence(df: pd.DataFrame, min_gradient: float = 1.0):
    """
    Analyseert de relatie tussen stijgingspercentage (%) en stapfrequentie (SPM).
    Berekent via lineaire regressie de daling in cadans per 10% extra helling.
    """
    df_work = df.copy()

    # 1. Automatisch hoogte- en afstandskolommen opsporen
    alt_col = next((c for c in ['altitude', 'enhanced_altitude'] if c in df_work.columns), None)
    dist_col = next((c for c in ['distance'] if c in df_work.columns), None)

    # 2. Hellingspercentage berekenen indien nog niet aanwezig
    if 'gradient' not in df_work.columns:
        if alt_col and dist_col:
            d_alt = df_work[alt_col].diff()
            d_dist = df_work[dist_col].diff()
            
            # Voorkom delen door 0 bij stilstand
            d_dist = d_dist.replace(0, np.nan)
            df_work['gradient'] = (d_alt / d_dist) * 100
        else:
            print(f"Beschikbare kolommen in dataframe: {list(df_work.columns)}")
            raise KeyError(f"Geen geschikte hoogte/afstand kolommen gevonden. Gevonden hoogte: {alt_col}, Afstand: {dist_col}")

    # Outliers en ruis in hellingspercentage opschonen
    df_work['gradient'] = df_work['gradient'].clip(lower=-30, upper=45)

    # Filter op klimpassages vanaf min_gradient
    df_climb = df_work[df_work['gradient'] >= min_gradient].copy()

    # 3. Cadans-kolom zoeken (cadence / steps / running_cadence / fractionall_cadence)
    cad_col = next((c for c in ['cadence', 'steps', 'running_cadence'] if c in df_climb.columns), None)

    if not cad_col:
        raise KeyError(f"Geen cadans-kolom gevonden. Beschikbare kolommen: {list(df_climb.columns)}")

    # Omrekenen naar totale stappen per minuut (SPM) indien enkelzijdig
    if df_climb[cad_col].median() < 100:
        df_climb['spm'] = df_climb[cad_col] * 2
    else:
        df_climb['spm'] = df_climb[cad_col]

    # Filter op geldige wandel/loopcadans
    df_climb = df_climb[(df_climb['spm'] > 40) & (df_climb['spm'] < 220)].dropna(subset=['gradient', 'spm'])

    # 4. Lineaire Regressie (Helling vs. Cadans)
    slope_coef, intercept = np.polyfit(df_climb['gradient'], df_climb['spm'], 1)
    cadence_drop_per_10pct = abs(slope_coef * 10)

    # 5. Visualisatie
    apply_triathlon_style()
    fig, ax = plt.subplots(figsize=(10, 6), dpi=300)

    fig.patch.set_facecolor(KLEUREN_PALETTE.get('background', '#000000'))
    ax.set_facecolor(KLEUREN_PALETTE.get('card_bg', '#1E222D'))

    # Colorbar op basis van de gevonden hoogtekolom
    color_array = df_climb[alt_col] if alt_col else '#00D2FF'

    scatter = ax.scatter(
        df_climb['gradient'], 
        df_climb['spm'], 
        c=color_array, 
        cmap='viridis', 
        alpha=0.6, 
        edgecolors='none', 
        s=25,
        label='Datapunten (1s)'
    )

    # Trendlijn tekenen
    x_vals = np.array([df_climb['gradient'].min(), df_climb['gradient'].max()])
    y_vals = intercept + slope_coef * x_vals
    ax.plot(x_vals, y_vals, color='#FF2A6D', linewidth=2.5, label=f'Trendlijn ({slope_coef:.2f} spm / +1%)')

    text_color = KLEUREN_PALETTE.get('text', '#FFFFFF')
    ax.set_xlabel('Hellingspercentage (%)', fontweight='bold', color=text_color)
    ax.set_ylabel('Cadans (Stappen per minuut - SPM)', fontweight='bold', color=text_color)
    ax.set_title('ETAPPE PERFORMANCE: TERREIN IMPACT — HELLING vs. CADANS', fontweight='bold', pad=15, color=text_color)

    if alt_col:
        cbar = plt.colorbar(scatter, ax=ax)
        cbar.set_label('Hoogte (m)', fontweight='bold', color=text_color)
        cbar.ax.yaxis.set_tick_params(color=text_color)
        plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color=text_color)

    # Statsoverlay
    stats_text = (
        f"• Cadansverlies per 10% stijging: -{cadence_drop_per_10pct:.1f} spm\n"
        f"• Gem. Klimcadans: {int(df_climb['spm'].mean())} spm\n"
        f"• Max. Helling geanalyseerd: {df_climb['gradient'].max():.1f}%"
    )

    ax.text(
        0.03, 0.95, stats_text, 
        transform=ax.transAxes, 
        fontsize=10, color='white', fontweight='bold',
        verticalalignment='top',
        bbox=dict(boxstyle='round,pad=0.6', facecolor='#111827', alpha=0.85, edgecolor='#374151')
    )

    ax.grid(True, linestyle='--', alpha=0.2)
    plt.tight_layout()

    return fig, cadence_drop_per_10pct

In [0]:
# Het FIT-bestand van etappe 2
fit_file_path = f"{DEFAULT_PATHS.fit_dir}/20260913_stubaier_hohenweg_etappe_2.fit"

# 1. Inlezen & Opschonen via de utils
df = parse_fit(fit_file_path)
df_clean = clean_fit_data(df)
df_active = calculate_elapsed_time(df_clean)

# 2. Genereer de grafiek en haal de KPI op
fig, drop_spm = analyze_slope_vs_cadence(df_active)

# 3. Toon de KPI direct in de notebook output
print(f" Resultaat Etappe 2: Je cadans daalt met {drop_spm:.1f} SPM per 10% stijging.")

# 4. Exporteer naar landscape formaat voor Instagram
export_for_instagram(fig, filename="20260913_slope_vs_cadence_etappe_2", format_type='landscape')